In [64]:
from sklearn.datasets import load_iris
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("iris.csv")

# 0 means setosa 
# 1 means Versicolor
# 2 means Virginica

In [65]:
# logistic regression without feature engineering 

X = df[["sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)"]]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 

Scaler = StandardScaler() 

X_train_scaled = Scaler.fit_transform(X_train)
X_test_scaled = Scaler.fit_transform(X_test)

model1 = LogisticRegression()

model1_cross_score = cross_val_score(model1, X_train_scaled, y_train, cv=15)

print(f"model1 cross val score: {model1_cross_score.mean()}")

model1.fit(X_train_scaled, y_train)

print(f"model1 score: {model1.score(X_test_scaled, y_test)}") #what this for, whats score??? 

y_pred = model1.predict(X_test_scaled)

print(f"model1 confusion matrix: {confusion_matrix(y_test, y_pred)}")
print(f"{classification_report(y_test, y_pred)}")

model1 cross val score: 0.9583333333333334
model1 score: 0.9666666666666667
model1 confusion matrix: [[10  0  0]
 [ 0  9  0]
 [ 0  1 10]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.90      1.00      0.95         9
           2       1.00      0.91      0.95        11

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



In [66]:
# feature engineering part but wiht logistic regression only 

df["sepal length (cm) / sepal width (cm)"] = df["sepal length (cm)"]/df["sepal width (cm)"]
df["petal length (cm) / petal width (cm)"] = df["petal length (cm)"]/df["petal width (cm)"]

# train test split 
X = df[["sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)", "sepal length (cm) / sepal width (cm)", "petal length (cm) / petal width (cm)"]]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_scaled = Scaler.fit_transform(X_train)
X_test_scaled = Scaler.transform(X_test)

model1.fit(X_train_scaled, y_train)

model1_feature_cross_score = cross_val_score(model1, X_train_scaled, y_train, cv=15)
print(f"{model1_feature_cross_score.mean()}")

y_pred_feature = model1.predict(X_test_scaled)

print(f"{confusion_matrix(y_test, y_pred_feature)}")
print(f"{classification_report(y_test, y_pred_feature)}")

0.9416666666666667
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [74]:
# same data but now knn 
df_knn = df.copy()
df_knn.drop(columns= ["sepal length (cm) / sepal width (cm)", "petal length (cm) / petal width (cm)"], inplace=True)


X_knn = df_knn[["sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)"]]
y_knn = df_knn["target"]

X_knn_train, X_knn_test, y_knn_train, y_knn_test = train_test_split(X_knn, y_knn, test_size=0.2, random_state=42) 

knn_pipe = Pipeline([
    ("Scaler", StandardScaler()),
    ("model_knn", KNeighborsClassifier(n_neighbors=5))
]) 

cross_val = cross_val_score(knn_pipe,X_knn_train,y_knn_train, cv=5 )
print(f"{cross_val.mean()}")

knn_pipe.fit(X_knn_train, y_knn_train)
y_pred_knn = knn_pipe.predict(X_knn_test)
print(y_pred_knn.tolist())
print(y_knn_test.tolist())
print(f"{confusion_matrix(y_knn_test,y_pred_knn)}")

0.9333333333333333
[1, 0, 2, 1, 1, 0, 1, 2, 1, 1, 2, 0, 0, 0, 0, 1, 2, 1, 1, 2, 0, 2, 0, 2, 2, 2, 2, 2, 0, 0]
[1, 0, 2, 1, 1, 0, 1, 2, 1, 1, 2, 0, 0, 0, 0, 1, 2, 1, 1, 2, 0, 2, 0, 2, 2, 2, 2, 2, 0, 0]
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]


In [75]:
# same data but now decision tree 

X_tree = df_knn[["sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)"]]
y_tree = df_knn["target"]

X_tree_train, X_tree_test, y_tree_train, y_tree_test = train_test_split(X_tree, y_tree, test_size=0.2, random_state=42) 

tree_pipe = Pipeline([
    ("Scaler", StandardScaler()),
    ("pca", PCA()),
    ("model_tree", DecisionTreeClassifier())
]) 

cross_val = cross_val_score(tree_pipe,X_tree_train,y_tree_train, cv=5 )
print(f"{cross_val.mean()}")

tree_pipe.fit(X_tree_train, y_tree_train)
y_pred_tree = tree_pipe.predict(X_tree_test)
print(y_pred_knn.tolist())
print(y_knn_test.tolist())
print(f"{confusion_matrix(y_knn_test,y_pred_knn)}")

0.9333333333333333
[1, 0, 2, 1, 1, 0, 1, 2, 1, 1, 2, 0, 0, 0, 0, 1, 2, 1, 1, 2, 0, 2, 0, 2, 2, 2, 2, 2, 0, 0]
[1, 0, 2, 1, 1, 0, 1, 2, 1, 1, 2, 0, 0, 0, 0, 1, 2, 1, 1, 2, 0, 2, 0, 2, 2, 2, 2, 2, 0, 0]
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]
